In [3]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import os

TASK_FILE = "guests.csv"
STATE_FILE = "guests_state.csv"

df = pd.read_csv(TASK_FILE)

# Combine names
df["Guest"] = df["first name"].astype(str) + " " + df["last name"].astype(str)

# Create state file if it doesn't exist
if not os.path.exists(STATE_FILE):
    state_df = df[["Guest"]].copy()
    state_df["Card_Written"] = False
    state_df.to_csv(STATE_FILE, index=False)

# Load saved state
state_df = pd.read_csv(STATE_FILE)

checkboxes = []

# Progress bar
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=len(state_df),
    description='Cards:',
    bar_style='success'
)

progress_label = widgets.Label()

def update_progress(change=None):

    checked = sum(cb.value for cb in checkboxes)

    progress.value = checked
    progress_label.value = f"{checked} / {len(checkboxes)} cards written"

    state_df["Card_Written"] = [cb.value for cb in checkboxes]
    state_df.to_csv(STATE_FILE, index=False)

# Create checkboxes
for _, row in state_df.iterrows():

    cb = widgets.Checkbox(
        value=row["Card_Written"],
        description=row["Guest"],
        indent=False
    )

    cb.observe(update_progress, names='value')

    checkboxes.append(cb)

update_progress()

display(
    widgets.VBox([
        progress,
        progress_label,
        widgets.HTML("<hr>"),
        widgets.VBox(checkboxes)
    ])
)

In [4]:
app_code = """
import streamlit as st
import pandas as pd
import os

df = pd.read_csv("guests.csv")
df["Guest"] = df["first name"] + " " + df["last name"]

STATE_FILE = "guests_state.csv"

if os.path.exists(STATE_FILE):
    state = pd.read_csv(STATE_FILE)
else:
    state = df[["Guest"]].copy()
    state["Card_Written"] = False
    state.to_csv(STATE_FILE, index=False)

st.title("Guest Place Card Progress")

checked = []

for i, row in state.iterrows():
    val = st.checkbox(
        row["Guest"],
        value=row["Card_Written"],
        key=f"checkbox_{i}"
    )
    checked.append(val)

state["Card_Written"] = checked
state.to_csv(STATE_FILE, index=False)

done = sum(checked)
total = len(checked)

st.progress(done / total if total else 0)
st.write(f"{done}/{total} written")
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully")

app.py created successfully


In [19]:
app_code = """
import streamlit as st
import pandas as pd
import os
from PIL import Image
import streamlit as st

PASSWORD = "GuestList"

st.title("Guest App Login")

password = st.text_input("Enter password", type="password")

if password != PASSWORD:
    st.warning("Incorrect password or no access.")
    st.stop()

# -------------------------
# MOBILE CONFIG
# -------------------------
st.set_page_config(
    page_title="Guest Tracker",
    layout="centered"
)

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_csv("guests.csv")
df.columns = df.columns.str.strip()

df["Guest"] = df["first name"].astype(str) + " " + df["last name"].astype(str)

# -------------------------
# STORAGE
# -------------------------
STATE_FILE = "guests_state.csv"
UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# -------------------------
# LOAD OR INIT STATE
# -------------------------
if os.path.exists(STATE_FILE):
    state = pd.read_csv(STATE_FILE)
else:
    state = df[["Guest"]].copy()
    state["Card_Written"] = False
    state["Approved"] = False
    state["Image_Path"] = ""
    state.to_csv(STATE_FILE, index=False)

st.title("📋 Guest Tracker")

# -------------------------
# PROGRESS (BASED ON WRITTEN)
# -------------------------
done = state["Card_Written"].sum()
total = len(state)

st.progress(done / total if total else 0)
st.write(f"✍️ {done}/{total} written")

st.markdown("---")

updated_rows = []

# -------------------------
# MOBILE CARD LIST (NO SWIPE BUTTONS)
# -------------------------
for i, row in state.iterrows():

    st.subheader(f"👤 {row['Guest']}")

    safe_key = str(row["Guest"]).replace(" ", "_").replace(".", "")

    # -------------------------
    # IMAGE PATH SAFE
    # -------------------------
    image_path = row.get("Image_Path", "")
    if pd.isna(image_path):
        image_path = ""
    else:
        image_path = str(image_path)

    # -------------------------
    # WRITTEN (drives progress)
    # -------------------------
    written = st.checkbox(
        "✍️ Card Written",
        value=row["Card_Written"],
        key=f"written_{i}_{safe_key}"
    )

    # -------------------------
    # UPLOAD (camera enabled on mobile)
    # -------------------------
    uploaded_file = st.file_uploader(
        "📸 Take / Upload Photo",
        type=["png", "jpg", "jpeg"],
        key=f"upload_{i}_{safe_key}"
    )

    if uploaded_file is not None:
        image_path = os.path.join(
            UPLOAD_DIR,
            f"{i}_{safe_key}_{uploaded_file.name}"
        )

        with open(image_path, "wb") as f:
            f.write(uploaded_file.getbuffer())

    # -------------------------
    # IMAGE PREVIEW (FULL WIDTH)
    # -------------------------
    if image_path and os.path.exists(image_path):
        st.image(Image.open(image_path), use_container_width=True)

    # -------------------------
    # APPROVED (UNDER PHOTO)
    # -------------------------
    approved = st.checkbox(
        "✔ Approved",
        value=row["Approved"],
        key=f"approved_{i}_{safe_key}"
    )

    # -------------------------
    # SAVE STATE (AUTO)
    # -------------------------
    updated_rows.append({
        "Guest": row["Guest"],
        "Card_Written": written,
        "Approved": approved,
        "Image_Path": image_path
    })

    st.markdown("---")

# -------------------------
# WRITE BACK (AUTO SAVE)
# -------------------------
state = pd.DataFrame(updated_rows)
state.to_csv(STATE_FILE, index=False)
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully")

app.py created successfully
